In [5]:
# À exécuter dans ton terminal si ce n'est pas déjà fait :
# pip install tiktoken sentencepiece transformers datasets accelerate torch

import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print("✅ Bibliothèques importées.")

✅ Bibliothèques importées.


In [10]:
from transformers import Qwen2Tokenizer, AutoModelForCausalLM
import torch

MODEL_REPO = "yannumber1/qwen-0.5b-gridglyph-atomic"

print(f"📡 Récupération du modèle : {MODEL_REPO}")

try:
    # On force l'utilisation de la classe Qwen2 qui est le backbone de ton modèle
    tokenizer = Qwen2Tokenizer.from_pretrained(
        MODEL_REPO, 
        trust_remote_code=True
    )
except Exception as e:
    print(f"⚠️ Erreur avec Qwen2Tokenizer : {e}")
    print("Tentative de secours avec chargement générique sans mode fast...")
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_REPO, 
        use_fast=False, 
        trust_remote_code=True
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO, 
    torch_dtype=torch.float16, 
    device_map="auto",
    trust_remote_code=True
)

print("✅ Modèle et Tokenizer chargés avec succès.")

📡 Récupération du modèle : yannumber1/qwen-0.5b-gridglyph-atomic


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 336/336 [00:00<00:00, 5420.16it/s]

✅ Modèle et Tokenizer chargés avec succès.


In [7]:
SEED_DATASET = "yannumber1/gridglyph-atomic-seeds"

print(f"📦 Chargement des seeds : {SEED_DATASET}")
dataset = load_dataset(SEED_DATASET, split="train")
df_test = dataset.to_pandas()

print(f"✅ Dataset chargé : {len(df_test)} exemples disponibles.")

📦 Chargement des seeds : yannumber1/gridglyph-atomic-seeds


Generating train split: 100%|██████████| 1602/1602 [00:00<00:00, 114949.79 examples/s]

✅ Dataset chargé : 1602 exemples disponibles.


In [48]:
import json
import random
import numpy as np

def apply_dynamic_iso(grid):
    # On utilise des lettres simples : A, B, C... pour les couleurs 0-9
    # C'est du texte pur, 100% compatible avec n'importe quel tokenizer.
    letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]
    perm_map = dict(zip(range(10), letters))
    
    if isinstance(grid, (np.ndarray, list)):
        return [[perm_map.get(int(cell), str(cell)) for cell in row] for row in grid]
    return grid

print("✅ Mapping ASCII (A-J) activé pour garantir la tokenisation.")

✅ Mapping ASCII (A-J) activé pour garantir la tokenisation.


In [49]:
def predict_dsl_atomic(row):
    # 1. Préparation du prompt
    iso_input = apply_dynamic_iso(row['input_grid'])
    iso_output = apply_dynamic_iso(row['output_grid'])
    
    in_grid_str = json.dumps(iso_input, ensure_ascii=False)
    out_grid_str = json.dumps(iso_output, ensure_ascii=False)

    prompt = (
        f"<|im_start|>system\nYou are a symbolic logic architect. Given a transformation, identify the underlying DSL rule.<|im_end|>\n"
        f"<|im_start|>user\nInput Grid:\n{in_grid_str}\n\nOutput Grid:\n{out_grid_str}\n\nWhat is the rule?<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    # 2. Encodage avec sécurité
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 3. Récupération du Stop Token (avec fallback)
    try:
        stop_ids = tokenizer.encode("<|im_end|>", add_special_tokens=False)
        stop_token_id = stop_ids[0] if stop_ids else tokenizer.eos_token_id
    except:
        stop_token_id = tokenizer.eos_token_id

    # 4. Génération
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,       # Un DSL est court, 64 suffisent largement
            temperature=0.01,
            do_sample=False,         # Mode pur logique
            eos_token_id=stop_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # 5. Décodage de la réponse uniquement
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    prediction = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Nettoyage si le modèle a quand même écrit <|im_end|> en texte
    return prediction.replace("<|im_end|>", "").strip()

print("✅ Moteur d'inférence paré à l'échec de tokenisation.")

✅ Moteur d'inférence paré à l'échec de tokenisation.


In [50]:
test_results = []
num_samples = 5 

samples = df_test.sample(min(num_samples, len(df_test)))

for idx, row in samples.iterrows():
    prediction = predict_dsl_atomic(row)
    
    test_results.append({
        'ground_truth': row['dsl_rule'],
        'prediction': prediction
    })

# Affichage des résultats
for i, res in enumerate(test_results):
    print(f"\n{'━'*40}")
    print(f"TEST {i+1}")
    print(f"EXPECTED : {res['ground_truth']}")
    print(f"MODEL    : {res['prediction']}")
    
    if res['prediction'].strip() == res['ground_truth'].strip():
        print("✨ SUCCESS: Logical match!")
    else:
        print("❌ MISMATCH")

RuntimeError: cannot reshape tensor of 0 elements into shape [1, 0, -1, 64] because the unspecified dimension size -1 can be any value and is ambiguous

  input_grid dsl_rule output_grid
0      [[1]]        ↔       [[1]]
